In [16]:
import pandas as pd
import numpy as np
import re
import nltk
import pickle
import html
import unicodedata

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse as sp

nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /Users/mix/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/mix/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [17]:
def create_stem_cache(corpus):
    unique_words = set()
    for text in corpus:
        text = str(text).lower()
        text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
        text = re.sub(r'[^a-z\s]', ' ', text)
        unique_words.update(text.split())
    
    stem_cache = {}
    ps = PorterStemmer()
    for w in unique_words:
        stem_cache[w] = ps.stem(w)
    return stem_cache

In [18]:
class CustomPreprocessor:
    def __init__(self, stop_dict, stem_cache):
        self.stop_dict = stop_dict
        self.stem_cache = stem_cache

    def __call__(self, s):
        s = str(s).lower()
        s = re.sub(r'^c\((.*)\)$', r'\1', s)
        s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8')
        s = re.sub(r'[^a-z\s]', ' ', s)
        tokens = s.split()
        tokens = [w for w in tokens if w not in self.stop_dict and len(w) > 2]
        tokens = [self.stem_cache.get(w, w) for w in tokens]
        return ' '.join(tokens)

In [19]:
class BM25Transformer:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b

    def fit(self, X):
        self.N_ = X.shape[0]
        df = np.bincount(X.indices, minlength=X.shape[1])
        self.idf_ = np.log((self.N_ - df + 0.5) / (df + 0.5) + 1.0)
        self.avgdl_ = X.sum(axis=1).mean()
        return self

    def transform(self, X):
        X = X.tocoo()
        dl = X.sum(axis=1).A1
        doc_lengths = dl[X.row]
        tf = X.data
        idf = self.idf_[X.col]
        numerator = tf * (self.k1 + 1)
        denominator = tf + self.k1 * (1 - self.b + self.b * (doc_lengths / self.avgdl_))
        data = (numerator / denominator) * idf
        return sp.csr_matrix((data, (X.row, X.col)), shape=X.shape)

In [20]:
class RecipeSearchEngine:
    def __init__(self, vectorizer, bm25_matrix, df):
        self.vectorizer = vectorizer
        self.bm25_matrix = bm25_matrix
        self.df = df

    def search(self, query, top_k=5):
        query_vec = self.vectorizer.transform([query])
        scores = self.bm25_matrix.dot(query_vec.T).toarray().flatten()
        rank = np.argsort(scores)[::-1]
        
        results = self.df.iloc[rank[:top_k]].copy()
        results['Score'] = scores[rank[:top_k]]
        return results

In [21]:
with open('resource/recipes.pkl', 'rb') as f:
    recipes = pickle.load(f)

In [22]:
recipes["Name"] = recipes["Name"].astype(str).apply(html.unescape)
recipes["RecipeIngredientParts"] = recipes["RecipeIngredientParts"].astype(str).apply(html.unescape)
recipes["RecipeInstructions"] = recipes["RecipeInstructions"].astype(str).apply(html.unescape)
recipes["Keywords"] = recipes["Keywords"].astype(str).apply(html.unescape)

recipes["Corpus"] = recipes["Name"] + " " + recipes["RecipeIngredientParts"] + " " + recipes["RecipeInstructions"] + " " + recipes["Keywords"]

In [23]:
stop_dict = set(stopwords.words('english'))
stem_cache = create_stem_cache(recipes['Corpus'])
my_custom_preprocessor = CustomPreprocessor(stop_dict, stem_cache)

In [24]:
count_vectorizer = CountVectorizer(preprocessor=my_custom_preprocessor)
term_counts = count_vectorizer.fit_transform(recipes['Corpus'])

In [25]:
bm25_transformer = BM25Transformer(k1=1.5, b=0.75)
bm25_transformer.fit(term_counts)
bm25_matrix = bm25_transformer.transform(term_counts)

In [ ]:
searcher = RecipeSearchEngine(count_vectorizer, bm25_matrix, recipes[['RecipeId', 'Name']], corpus=recipes['Corpus'].tolist())

with open('resource/recipe_search_engine.pkl', 'wb') as f:
    pickle.dump(searcher, f)

In [ ]:
with open('resource/recipe_search_engine.pkl', 'rb') as f:
    searcher = pickle.load(f)

# Test BM25 (default)
print("BM25 Search:")
results = searcher.search("eggs mix together", method='bm25')
display(results[['Name', 'Score']])

# Test TF-IDF
print("\nTF-IDF Search:")
results = searcher.search("eggs mix together", method='tfidf')
display(results[['Name', 'Score']])

# Test Exact Match
print("\nExact Match Search:")
results = searcher.search("eggs", method='exact')
display(results[['Name', 'Score']])

In [28]:
raw_text = recipes['Corpus'].iloc[0]
cleaned_text = my_custom_preprocessor(raw_text)
cleaned_text

'low fat berri blue frozen dessert blueberri granul sugar vanilla yogurt lemon juic toss cup berri sugar let stand minut stir occasion transfer berri sugar mixtur food processor add yogurt process smooth strain fine siev pour bake pan transfer ice cream maker process accord manufactur direct freez uncov edg solid centr soft transfer processor blend smooth return pan freez edg solid transfer processor blend smooth fold remain cup blueberri pour plastic mold freez overnight let soften slightli serv dessert low protein low cholesterol healthi free summer weeknight freezer easi'